# Lab 12 - K-Means Clustering: Impact of AI on Students

        Source dataset: `Datasets/Impact of AI on Students/ai_student_impact_dataset.csv`

        This notebook adapts the class lab pattern to the student-impact dataset. The source file is never modified.

        ## Lab concepts used

        - Cluster predictor-only student behaviour profiles.
- Select k with inertia and silhouette score.
- Interpret outcomes only after clusters are fitted.

        Interpretation is predictive and associative only. The Kaggle source does not document how the records were collected or whether they represent observed students.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative} from {Path.cwd()}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns from {DATA_PATH}")

In [ ]:
IDENTIFIER = "Student_ID"
OUTCOMES = ["Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"]
EARLY_RISK_FEATURES = [
    "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case",
    "Prompt_Engineering_Skill", "Tool_Diversity", "Paid_Subscription",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Institutional_Policy",
]
EXPANDED_FEATURES = EARLY_RISK_FEATURES + ["Anxiety_Level_During_Exams"]

df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)

assert IDENTIFIER not in EARLY_RISK_FEATURES
assert not set(OUTCOMES).intersection(EARLY_RISK_FEATURES)
print("Leakage policy ready. Primary burnout model excludes anxiety and all post-semester outcomes.")

## Predictor-only clustering matrix

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_features = [
    "Pre_Semester_GPA", "Weekly_GenAI_Hours", "Tool_Diversity",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Anxiety_Level_During_Exams",
]
sample = df.sample(n=min(12000, len(df)), random_state=RANDOM_STATE).copy()
scaled = StandardScaler().fit_transform(sample[cluster_features])

rows = []
for k in range(2, 8):
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = model.fit_predict(scaled)
    rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(
            scaled, labels, sample_size=min(5000, len(sample)),
            random_state=RANDOM_STATE
        ),
    })
scores = pd.DataFrame(rows)
display(scores)

In [ ]:
selected_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
final_kmeans = KMeans(
    n_clusters=selected_k, n_init=20, random_state=RANDOM_STATE
)
sample["Cluster"] = final_kmeans.fit_predict(scaled)
profile = sample.groupby("Cluster", observed=True)[
    cluster_features + ["GPA_Change", "Skill_Retention_Score"]
].mean().round(2)
burnout_mix = pd.crosstab(
    sample["Cluster"], sample["Burnout_Risk_Level"], normalize="index"
).round(3)
display(profile)
display(burnout_mix)

## What was learned from Lab 12

Outcomes were excluded from fitting, preventing the clusters from being defined by the answers later used for interpretation. K-means profiles are exploratory segments, not verified student types or causal groups.